In [12]:
# !pip install ultralytics

In [13]:
!pip install ultralytics torchinfo

In [14]:
import numpy as np
from dataclasses import dataclass
import os
import math
import shutil
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchinfo import summary as torch_summary
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import wandb
import time
import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import albumentations as A
import tqdm
# import fiftyone as fo

%matplotlib inline

In [15]:
from google.colab import drive
drive.mount('/content/drive')

In [16]:
wandb.login()

In [17]:
@dataclass
class Environment:
    env_name: str
    patches_yolo_config_yaml: str
    original_yolo_config_yaml: str
    patches_train_images_folder: str
    patches_train_labels_folder: str
    patches_val_images_folder: str
    patches_val_labels_folder: str
    original_val_images_folder: str
    original_val_labels_folder: str
    training_output_folder: str
    saved_weights_filepath: str
    video_input_filepath: str
    video_output_filepath: str
    device: str

    def fetch_original_val_ids(self):
        return np.array([Path(f).stem for f in os.listdir(self.original_val_images_folder)])

env: Environment = None
""" Set to environment-relevant config before training/inference """;

In [18]:
local_env = Environment(
    env_name='local',
    patches_yolo_config_yaml='data_patches_filtered/data.yaml',
    original_yolo_config_yaml='data/data.yaml',
    patches_train_images_folder='data_patches_filtered/train/images/',
    patches_train_labels_folder='data_patches_filtered/train/labels/',
    patches_val_images_folder='data_patches_filtered/val/images/',
    patches_val_labels_folder='data_patches_filtered/val/labels/',
    original_val_images_folder='data/val/images/',
    original_val_labels_folder='data/val/labels/',
    training_output_folder='data_gen/',
    saved_weights_filepath='data_gen/best.pt',
    video_input_filepath='data/inference_traffic_light_video.mp4',
    video_output_filepath='data_gen/traffic_light_detection_output_video.mp4',
    device='cpu',
)
kaggle_env = Environment(
    env_name='kaggle',
    patches_yolo_config_yaml='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/data.yaml',
    original_yolo_config_yaml='/kaggle/input/datasets/kyledunne/traffic-lights-dataset/data.yaml',
    patches_train_images_folder='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/train/images/',
    patches_train_labels_folder='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/train/labels/',
    patches_val_images_folder='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/val/images/',
    patches_val_labels_folder='/kaggle/input/datasets/kyledunne/traffic-lights-patches-dataset/val/labels/',
    original_val_images_folder='/kaggle/input/datasets/kyledunne/traffic-lights-dataset/val/images/',
    original_val_labels_folder='/kaggle/input/datasets/kyledunne/traffic-lights-dataset/val/labels/',
    training_output_folder='/kaggle/working/',
    saved_weights_filepath='/kaggle/input/datasets/kyledunne/traffic-lights-yolo-best-weights/best.pt',
    video_input_filepath='/kaggle/input/datasets/kyledunne/traffic-lights-dataset/inference_traffic_light_video.mp4',
    video_output_filepath='/kaggle/working/traffic_light_detection_output_video.mp4',
    device='cuda',
)
colab_home_folder = '/content/drive/My Drive/Colab Notebooks/traffic-light-detection/'
colab_env = Environment(
    env_name='colab',
    patches_yolo_config_yaml=colab_home_folder + 'data_patches_filtered/data.yaml',
    original_yolo_config_yaml=colab_home_folder + 'data/data.yaml',
    patches_train_images_folder=colab_home_folder + 'data_patches_filtered/train/images/',
    patches_train_labels_folder=colab_home_folder + 'data_patches_filtered/train/labels/',
    patches_val_images_folder=colab_home_folder + 'data_patches_filtered/val/images/',
    patches_val_labels_folder=colab_home_folder + 'data_patches_filtered/val/labels/',
    original_val_images_folder=colab_home_folder + 'data/val/images/',
    original_val_labels_folder=colab_home_folder + 'data/val/labels/',
    training_output_folder=colab_home_folder + 'data_gen/',
    saved_weights_filepath=colab_home_folder + 'data_gen/best.pt',
    video_input_filepath=colab_home_folder + 'data/inference_traffic_light_video.mp4',
    video_output_filepath=colab_home_folder + 'data_gen/traffic_light_detection_output_video.mp4',
    device='cuda',
)

In [19]:
class Config:
    def __init__(self, training, verbose=False):
        self.verbose = verbose
        self.training = training
        if self.training:
            os.makedirs(env.training_output_folder, exist_ok=True)

        self.class_labels = ['green', 'off', 'red', 'wait_on', 'yellow']

        self.seed = 8675309
        self.batch_size = 24
        self.starting_learning_rate = 0.001
        self.max_epochs = 500
        self.patience = 75
        self.num_workers = 8 if env.device == 'cuda' else 0
        self.pin_memory = self.num_workers > 0
        self.use_amp = env.device == 'cuda'

        self.original_image_width = 1920
        self.original_image_height = 1080

        self.model_name = 'yolo26x-p2.yaml'

        self.train_transforms = [
            A.RandomBrightnessContrast(brightness_limit=(-0.2, 0.2), contrast_limit=(-0.2, 0.2), p=0.3),
            A.CLAHE(clip_limit=(1, 4), tile_grid_size=(8, 8), p=0.2),
            A.RandomGamma(gamma_limit=(80, 120), p=0.2),
            A.PlanckianJitter(mode='blackbody', p=0.15),
            A.GaussianBlur(blur_limit=(3, 5), p=0.15),
            A.MotionBlur(blur_limit=(3, 5), p=0.1),
            A.GaussNoise(std_range=(0.03, 0.12), p=0.15),
            A.RandomFog(fog_coef_range=(0.1, 0.3), alpha_coef=0.08, p=0.1),
            A.RandomRain(brightness_coefficient=0.8, rain_type='drizzle', p=0.08),
            A.RandomShadow(shadow_roi=(0, 0, 1, 1), num_shadows_limit=(1, 3), shadow_intensity_range=(0.3, 0.5), p=0.1),
            A.ImageCompression(quality_range=(60, 95), p=0.15),
            A.Downscale(scale_range=(0.5, 0.9), p=0.1),
            A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.05, 0.2), p=0.1),
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.5, 1.0), p=0.1),
        ]

        self.hsv_h = 0.005
        self.hsv_s = 0.3
        self.hsv_v = 0.25

    def named_dict(self):
        return {
            'model': self.model_name,
            'epochs': self.max_epochs,
            'patience': self.patience,
            'seed': self.seed,
            'env': env.env_name,
            'device': env.device,
        }


config: Config = None
""" Set to environment-relevant config before training/inference """;

In [20]:
def plot_images_with_bounding_boxes(
        images,
        pred_bounding_boxes=None,
        pred_bounding_boxes_class_labels=None,
        pred_boxes_confidence=None,
        true_bounding_boxes=None,
        true_bounding_boxes_class_labels=None,
        white_text_background=True,
):
    for i, image in enumerate(images):
        h, w = image.shape[:2]
        fig, ax = plt.subplots(1)
        ax.imshow(image)
        ax.axis('off')

        if true_bounding_boxes is not None:
            for j, (x_center, y_center, bw, bh) in enumerate(true_bounding_boxes[i]):
                x0 = (x_center - bw / 2) * w
                y0 = (y_center - bh / 2) * h
                rect = patches.Rectangle((x0, y0), bw * w, bh * h,
                                         linewidth=1, edgecolor='orange', facecolor='none')
                ax.add_patch(rect)
                if true_bounding_boxes_class_labels is not None:
                    label = true_bounding_boxes_class_labels[i][j]
                    if white_text_background:
                        ax.text(x0, y0, label, color='orange', fontsize=8, va='bottom', ha='left',
                                bbox=dict(facecolor='white', edgecolor='none', pad=2))
                    else:
                        ax.text(x0, y0, label, color='orange', fontsize=8, va='bottom', ha='left')

        if pred_bounding_boxes is not None:
            for j, (x_center, y_center, bw, bh) in enumerate(pred_bounding_boxes[i]):
                x0 = (x_center - bw / 2) * w
                y0 = (y_center - bh / 2) * h
                rect = patches.Rectangle((x0, y0), bw * w, bh * h,
                                         linewidth=1, edgecolor='blue', facecolor='none')
                ax.add_patch(rect)
                if pred_boxes_confidence is not None:
                    conf_str = f"{int(pred_boxes_confidence[i][j] * 100)}%"
                    if pred_bounding_boxes_class_labels is not None:
                        label = f"{pred_bounding_boxes_class_labels[i][j]} {conf_str}"
                    else:
                        label = conf_str
                    if white_text_background:
                        ax.text(x0, y0, label, color='blue', fontsize=8,va='bottom', ha='left',
                                bbox=dict(facecolor='white', edgecolor='none', pad=2))
                    else:
                        ax.text(x0, y0, label, color='blue', fontsize=8,va='bottom', ha='left')

        plt.tight_layout()
        plt.show()

In [21]:
def plot_val_images_with_ground_truth_bounding_boxes(num_images_to_show=3, white_text_background=True):
    ids = np.random.choice(env.fetch_original_val_ids(), num_images_to_show)
    images = []
    all_boxes = []
    all_class_labels = []
    for image_id in ids:
        image_path = f'{env.original_val_images_folder}{image_id}.jpg'
        image_label = f'{env.original_val_labels_folder}{image_id}.txt'
        image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
        images.append(image)
        boxes = []
        box_labels = []
        with open(image_label, 'r') as label_file:
            for line in label_file:
                line_segments = line.strip().split()
                box_labels.append(config.class_labels[int(line_segments[0])])
                coords = [float(c) for c in line_segments[-4:]]
                boxes.append(coords)
        all_boxes.append(boxes)
        all_class_labels.append(box_labels)
    plot_images_with_bounding_boxes(images, true_bounding_boxes=all_boxes, true_bounding_boxes_class_labels=all_class_labels, white_text_background=white_text_background)

In [ ]:
def train_yolo():
    start_time = time.time()
    print(f't=0: Initializing training setup...')

    wandb.init(
        project='traffic-light-detection',
        name=f'run={int(start_time)}',
        config=config.named_dict(),
    )

    model = YOLO(config.model_name)

    def on_train_epoch_end(trainer):
        epoch = trainer.epoch + 1
        loss_items = trainer.label_loss_items(trainer.tloss, prefix='train')
        metrics = {}
        for key in ('train/box_loss', 'train/cls_loss', 'train/dfl_loss'):
            if key in loss_items:
                metrics[key] = loss_items[key]
        for key in ('lr/pg0', 'lr/pg1', 'lr/pg2'):
            if key in trainer.lr:
                metrics[key] = trainer.lr[key]
        if metrics:
            wandb.log(metrics, step=epoch)

    def on_fit_epoch_end(trainer):
        epoch = trainer.epoch + 1
        metrics = {}
        for key in ('metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss'):
            if key in trainer.metrics:
                metrics[key] = trainer.metrics[key]
        if metrics:
            wandb.log(metrics, step=epoch)

    model.add_callback('on_train_epoch_end', on_train_epoch_end)
    model.add_callback('on_fit_epoch_end', on_fit_epoch_end)

    results = model.train(
        data=env.original_yolo_config_yaml,
        epochs=config.max_epochs,
        patience=config.patience,
        imgsz=1280,
        rect=True,
        batch=24,
        optimizer='AdamW',
        cos_lr=False,
        seed=config.seed,
        amp=config.use_amp,
        workers=config.num_workers,
        device=env.device,
        verbose=True,
        plots=True,
        lr0 = 0.001,
        lrf = 0.2,
        warmup_epochs = 3.0,
        warmup_momentum = 0.8,
        warmup_bias_lr = 0.1,
        augmentations=config.train_transforms,
        hsv_h=config.hsv_h,
        hsv_s=config.hsv_s,
        hsv_v=config.hsv_v,
    )

    best_src = Path(results.save_dir) / 'weights' / 'best.pt'
    shutil.copy(str(best_src), env.saved_weights_filepath)
    print(f'Best model saved to {env.saved_weights_filepath}')

    artifact = wandb.Artifact(name='best-model', type='model')
    artifact.add_file(env.saved_weights_filepath)
    wandb.log_artifact(artifact)

    wandb.finish()
    return results

In [ ]:
env = colab_env
config = Config(training=True)
train_yolo()